### Ingest circuits.csv file

In [0]:
%run "../includes/configuration"
# run in individual cells

In [0]:
%run "../includes/common_functions"

##### Step 1 - Read the CSV file using the spark dataframe reader

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
circuits_schema = StructType(fields=[StructField("circuitId", IntegerType(), False),
                                     StructField("circuitRef", StringType(), True),
                                     StructField("name", StringType(), True),
                                     StructField("location", StringType(), True),
                                     StructField("country", StringType(), True),
                                     StructField("lat", DoubleType(), True),
                                     StructField("lng", DoubleType(), True),
                                     StructField("alt", IntegerType(), True),
                                     StructField("url", StringType(), True)])
circuits_df = spark.read \
  .option("header", True) \
  .schema(circuits_schema) \
  .csv(f"{raw_folder_path}/circuits.csv")

In [0]:
type(circuits_df)

In [0]:
circuits_df.show(5)

In [0]:
display(circuits_df.head(5))

In [0]:
circuits_df.printSchema()

In [0]:
circuits_df.describe().show()

#### Select only required columns

In [0]:
circuits_selected_df = circuits_df.select("circuitId", "circuitRef", "name", "location", "country", "lat", "lng", "alt")
display(circuits_selected_df)

In [0]:
# other way to select
circuits_selected_df = circuits_df.select(circuits_df.circuitId, circuits_df.circuitRef, circuits_df.name, circuits_df.location, circuits_df.country, circuits_df.lat, circuits_df.lng, circuits_df.alt)
display(circuits_selected_df)

In [0]:
# third way top select
circuits_selected_df = circuits_df.select(circuits_df["circuitId"], circuits_df["circuitRef"], circuits_df["name"], circuits_df["location"], circuits_df["country"], circuits_df["lat"], circuits_df["lng"], circuits_df["alt"])   
display(circuits_selected_df)

In [0]:
# last way to select
from pyspark.sql.functions import col
circuits_selected_df = circuits_df.select(col("circuitId"), col("circuitRef"), col("name"), col("location"), col("country"), col("lat"), col("lng"), col("alt"))
display(circuits_selected_df)

#### Rename columns

In [0]:
circuits_renamed_df = circuits_selected_df.withColumnRenamed("circuitId", "circuit_id") \
.withColumnRenamed("circuitRef", "circuit_ref") \
.withColumnRenamed("lat", "latitude") \
.withColumnRenamed("lng", "longitude") \
.withColumnRenamed("alt", "altitude")
display(circuits_renamed_df)

Add ingestion date to dataframe

In [0]:
# add current time stamp for each record
circuits_final_df = add_ingestion_date(circuits_renamed_df)

display(circuits_final_df)

#### Write data to datalake as parquet

In [0]:
circuits_final_df.write.mode('overwrite').parquet(f"{processed_folder_path}/circuits")

In [0]:
# read the data back to check
display(spark.read.parquet(f'{processed_folder_path}/circuits').head(5))
